# Creation of INTEGRAL orbit data in hdf5 format

---

## Notebook description

This notebook downloads position and velocity data from ISDC's website and compiles it into a HDF5 file, which will be used for further calculations.

#### REQUIRED INPUT:
 - None

#### OUTPUT:
 - HDF5 file with a table of INTEGRAL's XYZ vector positions and distance from Earth [`orbit_data.h5`]

In [2]:
import gzip
import shutil
from pathlib import Path

import requests

Download the official data from ISDC's website.

Running the cell below will download files (485 MiB) of files to your computer, supposing you have not downloaded them yet.

In [3]:
# http://isdcarc.unige.ch/arc/rev_3/aux/adp/XXXX.001/orbit_historic.fits.gz where XXXX is orbit name in 0001 - 2878 format
chunk_size = 256

dirpath = Path("./irem_orbit_data")
dirpath.mkdir(exist_ok=True)

numbers = [f"{i:04d}" for i in range(1, 2879)] # 0001 to 2878 inclusive
urls = [f"http://isdcarc.unige.ch/arc/rev_3/aux/adp/{number}.001/orbit_historic.fits.gz" for number in numbers]

filenames = []

for i, url in enumerate(urls, start=1):

    #convert 1 to 0001, 2 to 0002, ..., 2878 to 2878
    number = f"{i:04d}"

    filename = f"orbit_historic_{number}.fits.gz"
    filenames.append(filename)
    
    gz_path = dirpath / filename
    fits_path = dirpath / filename.replace('.gz', '')

    if fits_path.exists():
        #print(f"Skipping {fits_path.name} (already extracted).")
        continue

    if not gz_path.exists():
        #print(f"Downloading {filename}...")
        #print(url)
        req = requests.get(url, stream=True, verify=False)

        try:
            with open(gz_path, "wb") as file:
                for chunk in req.iter_content(chunk_size=chunk_size):
                    if chunk:
                        file.write(chunk)
            #print(f"Download successful: {filename}")
        except OSError as e:
            print(f"Download failed for {filename}: {e}")
            continue

    if not fits_path.exists():
        try:
            with gzip.open(gz_path, 'rb') as f_in:
                with open(fits_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            #print(f"Extracted {gz_path.name} -> {fits_path.name}")
        except Exception as e:
            print(f"Error extracting {gz_path}: {e}")

    

Read the data stored in FITS format and refactor it to fit the Pandas dataframe object:

In [4]:
from astropy.table import Table
import pandas as pd

df_list = []

for file in filenames:
    fits_path = dirpath / file.replace('.gz', '')
    if fits_path.exists():
        try:
            tbl = Table.read(fits_path)

            # build dict of scalar columns
            data = {}
            for name in tbl.colnames:
                col = tbl[name]
                if len(col.shape) == 1:
                    data[name] = col
                else:
                    # expand multidim into separate columns
                    for i in range(col.shape[1]):
                        data[f"{name}_{i}"] = col[:, i]

            new_df = pd.DataFrame(data)
            df_list.append(new_df)

            #print(f"Successfully read {fits_path.name}, number of rows: {len(tbl)}")

        except Exception as e:
            print(f"Error reading {fits_path}: {e}")
    else:
        print(f"File {fits_path.name} does not exist, skipping read.")

df = pd.concat(df_list, ignore_index=True)

Error reading irem_orbit_data/orbit_historic_0436.fits: Format could not be identified based on the file name or contents, please provide a 'format' argument.
The available formats are:
           Format           Read Write Auto-identify Deprecated
--------------------------- ---- ----- ------------- ----------
                      ascii  Yes   Yes            No           
               ascii.aastex  Yes   Yes            No           
                ascii.basic  Yes   Yes            No           
                  ascii.cds  Yes    No            No           
     ascii.commented_header  Yes   Yes            No           
                  ascii.csv  Yes   Yes           Yes           
              ascii.daophot  Yes    No            No           
                 ascii.ecsv  Yes   Yes           Yes           
           ascii.fast_basic  Yes   Yes            No           
ascii.fast_commented_header  Yes   Yes            No           
             ascii.fast_csv  Yes   Yes        

In [5]:
pd.options.display.max_columns = None
df.head(10)

,DAYBEG,DAYEND,EPOCH,ORBIN,SMAXIS,OMOTIN,NREC,XYZPOS_0,XYZPOS_1,XYZPOS_2,XYZVEL_0,XYZVEL_1,XYZVEL_2,RDIST,POLPOS_X_0,POLPOS_X_1,POLPOS_X_2,POLPOS_X_3,POLPOS_X_4,POLPOS_X_5,POLPOS_X_6,POLPOS_X_7,POLPOS_X_8,POLPOS_X_9,POLPOS_Y_0,POLPOS_Y_1,POLPOS_Y_2,POLPOS_Y_3,POLPOS_Y_4,POLPOS_Y_5,POLPOS_Y_6,POLPOS_Y_7,POLPOS_Y_8,POLPOS_Y_9,POLPOS_Z_0,POLPOS_Z_1,POLPOS_Z_2,POLPOS_Z_3,POLPOS_Z_4,POLPOS_Z_5,POLPOS_Z_6,POLPOS_Z_7,POLPOS_Z_8,POLPOS_Z_9,POLVEL_X_0,POLVEL_X_1,POLVEL_X_2,POLVEL_X_3,POLVEL_X_4,POLVEL_X_5,POLVEL_X_6,POLVEL_X_7,POLVEL_X_8,POLVEL_X_9,POLVEL_Y_0,POLVEL_Y_1,POLVEL_Y_2,POLVEL_Y_3,POLVEL_Y_4,POLVEL_Y_5,POLVEL_Y_6,POLVEL_Y_7,POLVEL_Y_8,POLVEL_Y_9,POLVEL_Z_0,POLVEL_Z_1,POLVEL_Z_2,POLVEL_Z_3,POLVEL_Z_4,POLVEL_Z_5,POLVEL_Z_6,POLVEL_Z_7,POLVEL_Z_8,POLVEL_Z_9
0,1020.238595,1020.240985,1020.239675,0.999,82492.75309,37527.94729,2,3787.349,2722.213,-5492.454,-4.747174,8.653850,2.894593,7205.658,0.026,0.008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.020,0.010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000061,0.000978,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000125,0.000737,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000049,0.000101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1020.240985,1020.243310,1020.242275,1.000,82810.33187,37744.86712,2,2626.242,4573.185,-4696.244,-5.549158,7.737529,4.173346,7061.578,0.013,-0.009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.022,-0.011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.014,-0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000121,0.000515,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000123,0.000826,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000003,0.000523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1020.243310,1020.245718,1020.244628,1.001,83249.07687,38045.23340,2,1446.615,6029.306,-3746.465,-6.007707,6.550210,5.124258,7244.392,0.002,-0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.008,-0.008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.022,-0.008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000041,0.000092,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000118,0.000317,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000074,0.000831,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1020.245718,1020.248368,1020.247149,1.002,83536.86714,38242.68613,2,114.007,7304.154,-2550.801,-6.175954,5.156311,5.790351,7737.586,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.010,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.020,-0.009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000007,-0.000006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000029,-0.000325,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000110,0.000693,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1020.248368,1020.251455,1020.250014,1.003,83555.38052,38255.39981,2,-1406.058,8396.535,-1067.169,-6.063765,3.710084,6.132586,8580.073,0.003,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.020,0.006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009,-0.006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000005,0.000090,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000051,-0.000580,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000078,0.000265,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1020.251455,1020.255236,1020.253449,1.004,83408.66112,38154.68197,2,-3157.991,9285.304,765.158,-5.720535,2.349236,6.165639,9837.441,0.006,-0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.018,0.007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.004,-0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000013,0.000142,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000067,-0.000438,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000021,-0.000087,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1020.255236,1020.260073,1020.257764,1.006,83248.30867,38044.70679,2,-5196.575,9925.508,3027.378,-5.216524,1.169732,5.943383,11605.391,0.005,-0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.010,0.005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.011,0.003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000017,0.000095,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000042,-0.000189,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000014,-0.000207,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1020.260073,1020.266483,1020.263398,1.008,83145.80715,37974.46321,2,-7587.120,10242.974,5823.125,-4.623670,0.219067,5.537671,14013.982,0.001,-0.001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.002,0.002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.012,0.004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0000

As you can see after running the cell above, it contains a lot of unnecesary data - mostly POS (Chebyshev's polynomials) for reconstruction of orbit data, which is sadly unavaiable for dates after 2003. We can therefore only get the data which is actually useful for us:

In [6]:
final_df = pd.DataFrame({
    "time": pd.to_datetime(df['EPOCH'], unit='d', origin="2000-01-01"),
    "x_j2000": df['XYZPOS_0'],
    "y_j2000": df['XYZPOS_1'],
    "z_j2000": df['XYZPOS_2'],
    "vx_j2000": df['XYZVEL_0'],
    "vy_j2000": df['XYZVEL_1'],
    "vz_j2000": df['XYZVEL_2'],
    "distance": df['RDIST'],
})

The following example showcases that this data source does contain ephemeris data in the 2022-2024 time range, unlike the official ESA kernels which we previously used:

In [7]:
final_df_july_2023 = final_df[(final_df['time'] >= '2023-07-01') & (final_df['time'] < '2023-08-01')]
final_df_july_2023.tail(1000)

,time,x_j2000,y_j2000,z_j2000,vx_j2000,vy_j2000,vz_j2000,distance
742443,2023-07-25 16:32:23.029648160,-17838.337,-13734.521,-3497.571,3.780675,2.454709,-3.124275,22783.246
742444,2023-07-25 16:34:05.960128005,-17445.970,-13479.374,-3818.499,3.843567,2.503217,-3.111225,22374.904
742445,2023-07-25 16:35:45.260166491,-17061.211,-13228.412,-4126.743,3.906188,2.551686,-3.096796,21979.668
742446,2023-07-25 16:37:21.107833596,-16683.840,-12981.532,-4422.822,3.968519,2.600099,-3.080994,21597.038
742447,2023-07-25 16:38:53.671350336,-16313.644,-12738.633,-4707.229,4.030541,2.648444,-3.063821,21226.534
...,...,...,...,...,...,...,...,...
743438,2023-07-31 19:22:20.451337795,-23419.059,-2185.929,130613.515,-0.655227,-0.401348,0.709546,132714.434
743439,2023-07-31 20:23:09.176992056,-25782.981,-3647.252,133056.456,-0.640446,-0.399528,0.630122,135580.548
743440,2023-07-31 21:26:31.538953726,-28188.140,-5161.567,135300.785,-0.624562,-0.396861,0.550936,138302.261
743441,2023-07-31 22:32:21.801471987,-30621.938,-6722.572,137320.200,-0.607580,-0.393349,0.472011,140853.589


Finally we save orbit data into the hdf5 format. You can optionally uncomment the line of code below and change the date range of the created hdf5 file:

In [7]:
#final_df = final_df[(final_df['time'] >= '2023-07-01') & (final_df['time'] < '2023-08-01')]


final_df.to_hdf('./orbit_data.h5',
              key="df",
              mode="w",
              format='table',
              complevel=1)